Metaclasse
- Cria classe de forma dinâmica.
- Pode ser útil na criação de frameworks onde as classes precisam ter um comportamento específico

In [14]:
class MeuMeta(type):
    def __new__(cls, nome, bases, dct):
        dct['novo_atributo'] = 'Valor adicionado pela metaclasse'
        return super().__new__(cls, nome, bases, dct)

In [15]:
class MinhaClasse(metaclass=MeuMeta):
    pass

In [16]:
objeto = MinhaClasse()
objeto.novo_atributo

'Valor adicionado pela metaclasse'

In [ ]:
# Definindo a metaclasse para validar atributos
class ValidadorMeta(type):
    def __new__(cls, nome, bases, dct):
        # Obtém o dicionário de validações (se existir) ou cria um vazio
        validacoes = dct.get('validacoes', {})

        # Itera sobre as validações definidas na classe
        for attr, tipo in validacoes.items():
            # Verifica se o tipo de validação é um tipo (não uma função)
            if not isinstance(tipo, type):  # Corrigido para checar tipo, não callable
                raise TypeError(f"O tipo de validação para {attr} deve ser um tipo válido.")

            # Define uma função de validação para cada atributo
            def valida_func(self, value, attr=attr, tipo=tipo):
                # Se o valor não for do tipo esperado, levanta um erro
                if not isinstance(value, tipo):
                    raise ValueError(f"{attr} deve ser do tipo {tipo.__name__}.")
                # Atribui o valor validado ao atri
                # buto da instância
                setattr(self, attr, value)

            # Renomeia a função para um nome único, baseado no nome do atributo
            valida_func.__name__ = f"set_{attr}"
            # Adiciona a função de validação ao dicionário da classe
            dct[f"set_{attr}"] = valida_func

        # Cria e retorna a classe com as funções de validação
        return super().__new__(cls, nome, bases, dct)

# Definindo a classe que usa a metaclasse ValidadorMeta
class Usuario(metaclass=ValidadorMeta):  # Aqui corrigido, metaclass é passado corretamente
    # Dicionário de validações dos atributos
    validacoes = {
        "nome": str,  # O atributo 'nome' deve ser do tipo 'str'
        "idade": int  # O atributo 'idade' deve ser do tipo 'int'
    }

    def __init__(self, nome, idade):
        # Chama as funções de validação para 'nome' e 'idade'
        # Isso valida se os tipos estão corretos e atribui os valores
        self.set_nome(nome)  # Chama a função gerada pela metaclasse para validar e atribuir 'nome'
        self.set_idade(idade)  # Chama a função gerada pela metaclasse para validar e atribuir 'idade'

# Testando a classe
try:
    u = Usuario("João", 30)  # Instancia um objeto com dados válidos
    print(u.nome, u.idade)  # Saída: João 30
    
    u_errado = Usuario("João", "trinta")  # Erro: 'idade' não é um int, gera um ValueError
except ValueError as e:
    print(e)  # Exibe a mensagem de erro


João 30
idade deve ser do tipo int.
